In [ ]:
import tensorflow as tf
import pandas as pd
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Download data
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

# Load data into DataFrames
df_train = pd.read_csv(train_file_path, sep="\t", header=None, names=['label', 'message'])
df_test = pd.read_csv(test_file_path, sep="\t", header=None, names=['label', 'message'])

# Convert labels to numbers (ham: 0, spam: 1)
df_train['label'] = df_train['label'].map({'ham': 0, 'spam': 1})
df_test['label'] = df_test['label'].map({'ham': 0, 'spam': 1})

train_labels = df_train.pop('label').values
test_labels = df_test.pop('label').values

train_sentences = df_train['message'].values
test_sentences = df_test['message'].values

In [ ]:
vocab_size = 1000
max_len = 50
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"

# Tokenization
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(train_sentences)

train_sequences = tokenizer.texts_to_sequences(train_sentences)
train_padded = pad_sequences(train_sequences, maxlen=max_len, padding=padding_type, truncating=trunc_type)

test_sequences = tokenizer.texts_to_sequences(test_sentences)
test_padded = pad_sequences(test_sequences, maxlen=max_len, padding=padding_type, truncating=trunc_type)

# Build Model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 16, input_length=max_len),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(train_padded, train_labels, epochs=10, validation_data=(test_padded, test_labels), verbose=2)

In [ ]:
def predict_message(pred_msg):
    # Convert message to sequence and pad it
    seq = tokenizer.texts_to_sequences([pred_msg])
    padded = pad_sequences(seq, maxlen=max_len, padding=padding_type, truncating=trunc_type)

    # Predict
    prediction = model.predict(padded)[0][0]

    # Determine label
    label = "spam" if prediction >= 0.5 else "ham"

    return [prediction, label]

# Example test
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
